# 18q — June 2026 ECMWF IFS single-run forecasts

This notebook reconstructs deterministic ECMWF IFS HRES forecast paths for the 30 certified June 2026 Hong Kong temperature event books.

It uses the same conservative information convention as the earlier deterministic weather ingestion:

- decision times are inherited from the verified 18p market panel;
- ECMWF run initialisation cycles are `00`, `06`, `12`, and `18` UTC;
- a run is treated as available only six hours after initialisation;
- the latest run satisfying `run_initialisation + 6 hours <= decision_time` is selected;
- forecasts are retrieved from Open-Meteo's Single Runs API;
- the requested model is `ecmwf_ifs`;
- latitude and longitude are fixed at `22.302219, 114.174637`;
- no elevation override is supplied;
- hourly two-metre temperature is returned in Hong Kong local time;
- the event-day maximum is the maximum across the 24 Hong Kong local hours.

The notebook produces:

- the 120 date-rule request plan;
- the 91 unique selected run requests implied by 30 consecutive dates;
- archived raw responses;
- run-level and request-level hourly temperature panels;
- deterministic event-day maximum forecasts;
- forecast errors against official HKO outcomes;
- deterministic contract-event membership, without a Gaussian bridge;
- integrity checks, issues, a report and a SHA-256 manifest.

Missing or unavailable runs are retained explicitly. No forecast is substituted from a later run, no observation is used as a forecast, and no Gaussian probability bridge is applied.

The notebook never creates a branch, commit, push, pull request, reminder or notification.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / ".git").exists():
    raise RuntimeError(
        "Run this notebook from the repository root. "
        f"Current directory: {REPO_ROOT}"
    )

STEP = "18q"
HKT = ZoneInfo("Asia/Hong_Kong")
UTC = timezone.utc

DECISION_PANEL_PATH = (
    REPO_ROOT
    / "data/processed/18p_june_2026_clob_market_price_recovery"
    / "18p_june_2026_no_lookahead_decision_panel.csv"
)
HKO_PATH = (
    REPO_ROOT
    / "data/processed/18o_june_2026_hko_realised_outcomes"
    / "18o_june_2026_hko_daily_max.csv"
)
CONTRACT_OUTCOME_PATH = (
    REPO_ROOT
    / "data/processed/18o_june_2026_hko_realised_outcomes"
    / "18o_june_2026_contract_outcomes.csv"
)

RAW_DIR = REPO_ROOT / "data/raw/18q_june_2026_ecmwf_single_run_forecasts"
OUT_DIR = (
    REPO_ROOT
    / "data/processed/18q_june_2026_ecmwf_single_run_forecasts"
)
REPORT_DIR = (
    REPO_ROOT
    / "reports/18q_june_2026_ecmwf_single_run_forecasts"
)

for directory in (RAW_DIR, OUT_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SINGLE_RUNS_URL = "https://single-runs-api.open-meteo.com/v1/forecast"

LATITUDE = 22.302219
LONGITUDE = 114.174637
MODEL = "ecmwf_ifs"
HOURLY_VARIABLE = "temperature_2m"
TIMEZONE_NAME = "Asia/Hong_Kong"
FORECAST_DAYS = 10
RUN_FREQUENCY_HOURS = 6
CONSERVATIVE_AVAILABILITY_LAG_HOURS = 6

REQUEST_TIMEOUT_SECONDS = 120
MAX_ATTEMPTS = 4
REQUEST_SLEEP_SECONDS = 0.10

DECISION_RULE_ORDER = [
    "24h_prior",
    "12h_prior",
    "6h_prior",
    "event_day_open",
]

ISSUE_COLUMNS = [
    "issue_level",
    "issue_code",
    "event_date",
    "decision_rule",
    "selected_run_initialisation_utc",
    "detail",
    "blocking",
]

session = requests.Session()
session.headers.update(
    {
        "User-Agent": (
            "2026MScWeatherForecastingPolymarket/"
            "18q-june-2026-ecmwf-single-run-forecasts"
        ),
        "Accept": "application/json",
    }
)

for required_input in (
    DECISION_PANEL_PATH,
    HKO_PATH,
    CONTRACT_OUTCOME_PATH,
):
    if not required_input.is_file():
        raise FileNotFoundError(
            f"Required verified input is missing: {required_input}"
        )

print(f"Repository root: {REPO_ROOT}")
print(f"Model: {MODEL}")
print(f"Coordinates: {LATITUDE}, {LONGITUDE}")
print(
    "Conservative availability lag: "
    f"{CONSERVATIVE_AVAILABILITY_LAG_HOURS} hours"
)

Repository root: /Users/edwardlee/Desktop/2026MScWeatherForecastingPolymarket
Model: ecmwf_ifs
Coordinates: 22.302219, 114.174637
Conservative availability lag: 6 hours


In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def as_utc_timestamp(value: Any) -> pd.Timestamp:
    return pd.to_datetime(value, utc=True)


def select_latest_admissible_run(
    decision_cutoff_utc: pd.Timestamp,
) -> pd.Timestamp:
    latest_initialisation = (
        decision_cutoff_utc
        - pd.Timedelta(hours=CONSERVATIVE_AVAILABILITY_LAG_HOURS)
    )
    latest_initialisation = latest_initialisation.floor(
        f"{RUN_FREQUENCY_HOURS}h"
    )
    return latest_initialisation


def run_key(run_initialisation_utc: pd.Timestamp) -> str:
    return run_initialisation_utc.strftime("%Y%m%dT%H%MZ")


def run_parameter(run_initialisation_utc: pd.Timestamp) -> str:
    return run_initialisation_utc.strftime("%Y-%m-%dT%H:%M")


def request_single_run(
    run_initialisation_utc: pd.Timestamp,
) -> tuple[dict[str, Any] | None, dict[str, Any]]:
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "hourly": HOURLY_VARIABLE,
        "models": MODEL,
        "run": run_parameter(run_initialisation_utc),
        "forecast_days": FORECAST_DAYS,
        "timezone": TIMEZONE_NAME,
        "temperature_unit": "celsius",
        "timeformat": "iso8601",
        "cell_selection": "land",
    }

    last_error = ""
    last_status: int | None = None
    last_url = SINGLE_RUNS_URL

    for attempt in range(1, MAX_ATTEMPTS + 1):
        requested_at = datetime.now(UTC)
        try:
            response = session.get(
                SINGLE_RUNS_URL,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            last_status = response.status_code
            last_url = response.url

            if response.status_code == 429 or response.status_code >= 500:
                last_error = f"HTTP {response.status_code}"
                time.sleep(1.5 * attempt)
                continue

            if response.status_code >= 400:
                error_text = response.text[:1000]
                return None, {
                    "requested_at_utc": requested_at.isoformat(),
                    "request_url": response.url,
                    "attempt": attempt,
                    "status_code": response.status_code,
                    "success": False,
                    "error": f"HTTP {response.status_code}: {error_text}",
                }

            payload = response.json()
            if not isinstance(payload, dict):
                raise ValueError(
                    f"Expected JSON object, found {type(payload).__name__}"
                )

            if payload.get("error") is True:
                return None, {
                    "requested_at_utc": requested_at.isoformat(),
                    "request_url": response.url,
                    "attempt": attempt,
                    "status_code": response.status_code,
                    "success": False,
                    "error": str(payload.get("reason", "API error")),
                }

            return payload, {
                "requested_at_utc": requested_at.isoformat(),
                "request_url": response.url,
                "attempt": attempt,
                "status_code": response.status_code,
                "success": True,
                "error": "",
            }

        except (
            requests.RequestException,
            ValueError,
            json.JSONDecodeError,
        ) as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            time.sleep(1.5 * attempt)

    return None, {
        "requested_at_utc": datetime.now(UTC).isoformat(),
        "request_url": last_url,
        "attempt": MAX_ATTEMPTS,
        "status_code": last_status,
        "success": False,
        "error": last_error or "Unknown request failure",
    }


def parse_hourly_payload(
    *,
    payload: dict[str, Any],
    selected_run_initialisation_utc: pd.Timestamp,
    raw_response_path: Path,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    hourly = payload.get("hourly")
    if not isinstance(hourly, dict):
        raise ValueError("Response does not contain an hourly object")

    times = hourly.get("time")
    temperatures = hourly.get(HOURLY_VARIABLE)

    if not isinstance(times, list) or not isinstance(temperatures, list):
        raise ValueError(
            "Hourly time or temperature_2m is not a list"
        )

    if len(times) != len(temperatures):
        raise ValueError(
            "Hourly time and temperature arrays have different lengths"
        )

    frame = pd.DataFrame(
        {
            "forecast_valid_time_local_text": times,
            "temperature_2m_c": temperatures,
        }
    )

    local_naive = pd.to_datetime(
        frame["forecast_valid_time_local_text"],
        errors="coerce",
    )
    frame["forecast_valid_time_hkt"] = local_naive.dt.tz_localize(
        HKT,
        ambiguous="raise",
        nonexistent="raise",
    )
    frame["forecast_valid_time_utc"] = (
        frame["forecast_valid_time_hkt"].dt.tz_convert("UTC")
    )
    frame["temperature_2m_c"] = pd.to_numeric(
        frame["temperature_2m_c"],
        errors="coerce",
    )
    frame["selected_run_initialisation_utc"] = (
        selected_run_initialisation_utc
    )
    frame["selected_run_key"] = run_key(
        selected_run_initialisation_utc
    )
    frame["lead_hours"] = (
        frame["forecast_valid_time_utc"]
        - selected_run_initialisation_utc
    ).dt.total_seconds() / 3600.0
    frame["raw_response_path"] = str(
        raw_response_path.relative_to(REPO_ROOT)
    )

    frame = (
        frame.sort_values("forecast_valid_time_utc")
        .drop_duplicates("forecast_valid_time_utc", keep="last")
        .reset_index(drop=True)
    )

    metadata = {
        "returned_latitude": payload.get("latitude"),
        "returned_longitude": payload.get("longitude"),
        "returned_elevation": payload.get("elevation"),
        "returned_timezone": payload.get("timezone"),
        "returned_timezone_abbreviation": payload.get(
            "timezone_abbreviation"
        ),
        "returned_utc_offset_seconds": payload.get(
            "utc_offset_seconds"
        ),
        "generationtime_ms": payload.get("generationtime_ms"),
        "hourly_units_temperature": (
            payload.get("hourly_units", {})
            .get(HOURLY_VARIABLE)
        ),
        "parsed_hourly_rows": len(frame),
        "parsed_nonmissing_temperature_rows": int(
            frame["temperature_2m_c"].notna().sum()
        ),
        "first_valid_time_utc": (
            frame["forecast_valid_time_utc"].min().isoformat()
            if not frame.empty
            else ""
        ),
        "last_valid_time_utc": (
            frame["forecast_valid_time_utc"].max().isoformat()
            if not frame.empty
            else ""
        ),
    }

    return frame, metadata


def deterministic_membership(
    temperature: float,
    event_type: str,
    lower_bound: float | None,
    upper_bound: float | None,
) -> int:
    if event_type == "lower":
        return int(temperature < float(upper_bound))
    if event_type == "interior":
        return int(
            float(lower_bound) <= temperature < float(upper_bound)
        )
    if event_type == "upper":
        return int(temperature >= float(lower_bound))
    raise ValueError(f"Unknown event type: {event_type}")

In [3]:
decision_panel = pd.read_csv(
    DECISION_PANEL_PATH,
    dtype={
        "market_id": str,
        "condition_id": str,
        "selected_yes_token_id": str,
    },
)
hko = pd.read_csv(HKO_PATH)
contract_outcomes = pd.read_csv(
    CONTRACT_OUTCOME_PATH,
    dtype={
        "market_id": str,
        "condition_id": str,
        "yes_token_id": str,
        "no_token_id": str,
    },
)

decision_panel["event_date"] = pd.to_datetime(
    decision_panel["event_date"]
)
decision_panel["decision_cutoff_utc"] = pd.to_datetime(
    decision_panel["decision_cutoff_utc"],
    utc=True,
)
hko["event_date"] = pd.to_datetime(hko["event_date"])
contract_outcomes["event_date"] = pd.to_datetime(
    contract_outcomes["event_date"]
)

if len(decision_panel) != 1320:
    raise AssertionError(
        f"Expected 1,320 verified 18p decision rows, "
        f"found {len(decision_panel)}"
    )
if decision_panel["event_date"].nunique() != 30:
    raise AssertionError("Expected 30 decision-panel dates")
if len(hko) != 30 or hko["event_date"].nunique() != 30:
    raise AssertionError("Expected 30 official HKO daily maxima")
if len(contract_outcomes) != 330:
    raise AssertionError("Expected 330 certified contract outcomes")
if not contract_outcomes.groupby(
    "event_date"
)["realised_yes"].sum().eq(1).all():
    raise AssertionError(
        "At least one contract book lacks exactly one winner"
    )

cutoff_consistency = (
    decision_panel.groupby(
        ["event_date", "decision_rule"],
        as_index=False,
    )
    .agg(
        n_contract_rows=("market_id", "size"),
        n_unique_cutoffs=(
            "decision_cutoff_utc",
            "nunique",
        ),
        decision_cutoff_utc=(
            "decision_cutoff_utc",
            "first",
        ),
    )
)

if len(cutoff_consistency) != 120:
    raise AssertionError(
        f"Expected 120 date-rule cutoff rows, "
        f"found {len(cutoff_consistency)}"
    )
if not cutoff_consistency["n_contract_rows"].eq(11).all():
    raise AssertionError(
        "At least one date-rule does not contain 11 contracts"
    )
if not cutoff_consistency["n_unique_cutoffs"].eq(1).all():
    raise AssertionError(
        "At least one date-rule has inconsistent decision cutoffs"
    )

request_plan = cutoff_consistency[
    [
        "event_date",
        "decision_rule",
        "decision_cutoff_utc",
    ]
].copy()
request_plan["decision_rule_order"] = request_plan[
    "decision_rule"
].map(
    {
        rule: index
        for index, rule in enumerate(DECISION_RULE_ORDER)
    }
)
if request_plan["decision_rule_order"].isna().any():
    raise AssertionError("Unexpected decision rule found")

request_plan["decision_cutoff_hkt"] = (
    request_plan["decision_cutoff_utc"].dt.tz_convert(HKT)
)
request_plan["selected_run_initialisation_utc"] = (
    request_plan["decision_cutoff_utc"].map(
        select_latest_admissible_run
    )
)
request_plan["selected_run_available_utc"] = (
    request_plan["selected_run_initialisation_utc"]
    + pd.Timedelta(
        hours=CONSERVATIVE_AVAILABILITY_LAG_HOURS
    )
)
request_plan["selected_run_key"] = request_plan[
    "selected_run_initialisation_utc"
].map(run_key)
request_plan["selected_run_parameter"] = request_plan[
    "selected_run_initialisation_utc"
].map(run_parameter)

event_start_hkt = request_plan["event_date"].map(
    lambda value: pd.Timestamp(
        year=value.year,
        month=value.month,
        day=value.day,
        hour=0,
        tz=HKT,
    )
)
event_end_hkt = event_start_hkt + pd.Timedelta(hours=23)

request_plan["event_start_hkt"] = event_start_hkt
request_plan["event_end_hkt"] = event_end_hkt
request_plan["event_start_utc"] = event_start_hkt.dt.tz_convert(
    "UTC"
)
request_plan["event_end_utc"] = event_end_hkt.dt.tz_convert(
    "UTC"
)
request_plan["lead_hours_to_event_start"] = (
    request_plan["event_start_utc"]
    - request_plan["selected_run_initialisation_utc"]
).dt.total_seconds() / 3600.0
request_plan["lead_hours_to_event_end"] = (
    request_plan["event_end_utc"]
    - request_plan["selected_run_initialisation_utc"]
).dt.total_seconds() / 3600.0
request_plan["run_admissible_at_cutoff"] = (
    request_plan["selected_run_available_utc"]
    <= request_plan["decision_cutoff_utc"]
)

run_hours = request_plan[
    "selected_run_initialisation_utc"
].dt.hour
valid_cycle_hours = {0, 6, 12, 18}

if len(request_plan) != 120:
    raise AssertionError("Request plan does not contain 120 rows")
if request_plan.duplicated(
    ["event_date", "decision_rule"]
).any():
    raise AssertionError("Duplicate date-rule request-plan keys")
if not request_plan[
    "run_admissible_at_cutoff"
].all():
    raise AssertionError(
        "At least one selected run is unavailable at its cutoff"
    )
if not set(run_hours.unique()).issubset(valid_cycle_hours):
    raise AssertionError(
        "Selected run initialisation is not a valid six-hour cycle"
    )

unique_runs = (
    request_plan[
        [
            "selected_run_initialisation_utc",
            "selected_run_key",
            "selected_run_parameter",
        ]
    ]
    .drop_duplicates()
    .sort_values("selected_run_initialisation_utc")
    .reset_index(drop=True)
)

if len(unique_runs) != 91:
    raise AssertionError(
        f"Expected 91 unique selected runs for 30 consecutive dates, "
        f"found {len(unique_runs)}"
    )

print(f"Request-plan rows: {len(request_plan)}")
print(f"Unique selected runs: {len(unique_runs)}")
display(
    request_plan[
        [
            "event_date",
            "decision_rule",
            "decision_cutoff_utc",
            "selected_run_initialisation_utc",
            "selected_run_available_utc",
            "lead_hours_to_event_start",
        ]
    ].head(12)
)

Request-plan rows: 120
Unique selected runs: 91


,event_date,decision_rule,decision_cutoff_utc,selected_run_initialisation_utc,selected_run_available_utc,lead_hours_to_event_start
0,2026-06-01,12h_prior,2026-05-31 04:00:00+00:00,2026-05-30 18:00:00+00:00,2026-05-31 00:00:00+00:00,22.0
1,2026-06-01,24h_prior,2026-05-30 16:00:00+00:00,2026-05-30 06:00:00+00:00,2026-05-30 12:00:00+00:00,34.0
2,2026-06-01,6h_prior,2026-05-31 10:00:00+00:00,2026-05-31 00:00:00+00:00,2026-05-31 06:00:00+00:00,16.0
3,2026-06-01,event_day_open,2026-05-31 16:00:00+00:00,2026-05-31 06:00:00+00:00,2026-05-31 12:00:00+00:00,10.0
4,2026-06-02,12h_prior,2026-06-01 04:00:00+00:00,2026-05-31 18:00:00+00:00,2026-06-01 00:00:00+00:00,22.0
5,2026-06-02,24h_prior,2026-05-31 16:00:00+00:00,2026-05-31 06:00:00+00:00,2026-05-31 12:00:00+00:00,34.0
6,2026-06-02,6h_prior,2026-06-01 10:00:00+00:00,2026-06-01 00:00:00+00:00,2026-06-01 06:00:00+00:00,16.0
7,2026-06-02,event_day_open,2026-06-01 16:00:00+00:00,2026-06-01 06:00:00+00:00,2026-06-01 12:00:00+00:00,10.0
8,2026-06-03,12h_prior,2026-06-02 04:00:00+00:00,2026-06-01 18:00:00+00:00,2026-06-02 00:00:00+00:00,22.0
9,2026-06-03,24h_prior,2026-06-01 16:00:00+00:00,2026-06-01 06:00:00+00:00,2026-06-01 12:00:00+00:00,34.0


In [4]:
issue_rows: list[dict[str, Any]] = []
fetch_rows: list[dict[str, Any]] = []
successful_hourly_frames: list[pd.DataFrame] = []

for run_number, run_row in enumerate(
    unique_runs.itertuples(index=False),
    start=1,
):
    selected_run = pd.Timestamp(
        run_row.selected_run_initialisation_utc
    )
    raw_path = (
        RAW_DIR
        / f"ecmwf_ifs_{run_key(selected_run)}.json"
    )

    payload, request_metadata = request_single_run(
        selected_run
    )

    archive_payload: dict[str, Any]
    parse_metadata: dict[str, Any] = {
        "returned_latitude": np.nan,
        "returned_longitude": np.nan,
        "returned_elevation": np.nan,
        "returned_timezone": "",
        "returned_timezone_abbreviation": "",
        "returned_utc_offset_seconds": np.nan,
        "generationtime_ms": np.nan,
        "hourly_units_temperature": "",
        "parsed_hourly_rows": 0,
        "parsed_nonmissing_temperature_rows": 0,
        "first_valid_time_utc": "",
        "last_valid_time_utc": "",
    }
    parse_success = False
    parse_error = ""

    if payload is None:
        archive_payload = {
            "archived_request_failure": True,
            "selected_run_initialisation_utc": (
                selected_run.isoformat()
            ),
            "request_metadata": request_metadata,
        }
    else:
        archive_payload = payload

    raw_path.write_text(
        json.dumps(
            archive_payload,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    if payload is not None:
        try:
            hourly_frame, parse_metadata = parse_hourly_payload(
                payload=payload,
                selected_run_initialisation_utc=selected_run,
                raw_response_path=raw_path,
            )
            parse_success = (
                not hourly_frame.empty
                and hourly_frame[
                    "temperature_2m_c"
                ].notna().any()
            )
            if parse_success:
                successful_hourly_frames.append(hourly_frame)
            else:
                parse_error = (
                    "Parsed response contains no non-missing "
                    "temperature rows"
                )
        except Exception as exc:
            parse_error = f"{type(exc).__name__}: {exc}"

    fetch_row = {
        "selected_run_initialisation_utc": selected_run,
        "selected_run_key": run_key(selected_run),
        "selected_run_parameter": run_parameter(selected_run),
        "model": MODEL,
        "latitude_requested": LATITUDE,
        "longitude_requested": LONGITUDE,
        "forecast_days": FORECAST_DAYS,
        "hourly_variable": HOURLY_VARIABLE,
        "timezone_requested": TIMEZONE_NAME,
        "availability_lag_hours": (
            CONSERVATIVE_AVAILABILITY_LAG_HOURS
        ),
        **request_metadata,
        "parse_success": parse_success,
        "parse_error": parse_error,
        **parse_metadata,
        "raw_response_path": str(
            raw_path.relative_to(REPO_ROOT)
        ),
        "raw_response_size_bytes": raw_path.stat().st_size,
        "raw_response_sha256": sha256_file(raw_path),
    }
    fetch_rows.append(fetch_row)

    if not request_metadata["success"]:
        issue_rows.append(
            {
                "issue_level": "unique_run",
                "issue_code": "RUN_FETCH_FAILED",
                "event_date": "",
                "decision_rule": "",
                "selected_run_initialisation_utc": (
                    selected_run.isoformat()
                ),
                "detail": request_metadata["error"],
                "blocking": False,
            }
        )
    elif not parse_success:
        issue_rows.append(
            {
                "issue_level": "unique_run",
                "issue_code": "RUN_PARSE_FAILED",
                "event_date": "",
                "decision_rule": "",
                "selected_run_initialisation_utc": (
                    selected_run.isoformat()
                ),
                "detail": parse_error,
                "blocking": False,
            }
        )

    if run_number % 10 == 0 or run_number == len(unique_runs):
        print(
            f"Processed {run_number}/{len(unique_runs)} unique runs; "
            f"successful={sum(bool(row['parse_success']) for row in fetch_rows)}"
        )

    time.sleep(REQUEST_SLEEP_SECONDS)

fetch_inventory = pd.DataFrame(fetch_rows)

run_hourly = pd.concat(
    successful_hourly_frames,
    ignore_index=True,
) if successful_hourly_frames else pd.DataFrame(
    columns=[
        "forecast_valid_time_local_text",
        "temperature_2m_c",
        "forecast_valid_time_hkt",
        "forecast_valid_time_utc",
        "selected_run_initialisation_utc",
        "selected_run_key",
        "lead_hours",
        "raw_response_path",
    ]
)

if not run_hourly.empty:
    run_hourly = (
        run_hourly.sort_values(
            [
                "selected_run_initialisation_utc",
                "forecast_valid_time_utc",
            ]
        )
        .drop_duplicates(
            [
                "selected_run_initialisation_utc",
                "forecast_valid_time_utc",
            ],
            keep="last",
        )
        .reset_index(drop=True)
    )

print(
    "Unique run fetch success: "
    f"{int(fetch_inventory['success'].sum())}/{len(fetch_inventory)}"
)
print(
    "Unique run parse success: "
    f"{int(fetch_inventory['parse_success'].sum())}/{len(fetch_inventory)}"
)
print(f"Unique-run hourly rows: {len(run_hourly)}")

Processed 10/91 unique runs; successful=10


Processed 20/91 unique runs; successful=20


Processed 30/91 unique runs; successful=30


Processed 40/91 unique runs; successful=40


Processed 50/91 unique runs; successful=50


Processed 60/91 unique runs; successful=60


Processed 70/91 unique runs; successful=70


Processed 80/91 unique runs; successful=79


Processed 90/91 unique runs; successful=89
Processed 91/91 unique runs; successful=90


Unique run fetch success: 91/91
Unique run parse success: 90/91
Unique-run hourly rows: 21600


In [5]:
hourly_by_run = {
    pd.Timestamp(run): group.copy()
    for run, group in run_hourly.groupby(
        "selected_run_initialisation_utc"
    )
}

request_hourly_frames: list[pd.DataFrame] = []
daily_rows: list[dict[str, Any]] = []

hko_lookup = (
    hko.set_index("event_date")["hko_daily_max_c"]
    .astype(float)
    .to_dict()
)

for request in request_plan.itertuples(index=False):
    event_date = pd.Timestamp(request.event_date)
    selected_run = pd.Timestamp(
        request.selected_run_initialisation_utc
    )
    run_frame = hourly_by_run.get(selected_run)

    if run_frame is None or run_frame.empty:
        selected = pd.DataFrame(
            columns=run_hourly.columns
        )
    else:
        selected = run_frame.loc[
            run_frame[
                "forecast_valid_time_hkt"
            ].dt.date
            == event_date.date()
        ].copy()

    selected = selected.sort_values(
        "forecast_valid_time_hkt"
    ).reset_index(drop=True)

    if not selected.empty:
        selected.insert(
            0,
            "event_date",
            event_date.date().isoformat(),
        )
        selected.insert(
            1,
            "decision_rule",
            request.decision_rule,
        )
        selected.insert(
            2,
            "decision_rule_order",
            int(request.decision_rule_order),
        )
        selected.insert(
            3,
            "decision_cutoff_utc",
            request.decision_cutoff_utc,
        )
        selected.insert(
            4,
            "decision_cutoff_hkt",
            request.decision_cutoff_hkt,
        )
        request_hourly_frames.append(selected)

    nonmissing = selected.loc[
        selected["temperature_2m_c"].notna()
    ].copy()

    n_rows = len(selected)
    n_nonmissing = len(nonmissing)
    n_unique_hours = (
        selected["forecast_valid_time_hkt"].nunique()
        if not selected.empty
        else 0
    )

    ready = (
        n_rows == 24
        and n_nonmissing == 24
        and n_unique_hours == 24
    )

    if ready:
        max_index = nonmissing[
            "temperature_2m_c"
        ].idxmax()
        max_row = nonmissing.loc[max_index]
        forecast_max = float(
            nonmissing["temperature_2m_c"].max()
        )
        forecast_min = float(
            nonmissing["temperature_2m_c"].min()
        )
        forecast_mean = float(
            nonmissing["temperature_2m_c"].mean()
        )
        max_time_hkt = pd.Timestamp(
            max_row["forecast_valid_time_hkt"]
        )
        max_time_utc = pd.Timestamp(
            max_row["forecast_valid_time_utc"]
        )
    else:
        forecast_max = np.nan
        forecast_min = np.nan
        forecast_mean = np.nan
        max_time_hkt = pd.NaT
        max_time_utc = pd.NaT

    hko_actual = float(hko_lookup[event_date])
    forecast_error = (
        forecast_max - hko_actual
        if ready
        else np.nan
    )

    daily_rows.append(
        {
            "event_date": event_date,
            "decision_rule": request.decision_rule,
            "decision_rule_order": int(
                request.decision_rule_order
            ),
            "decision_cutoff_utc": request.decision_cutoff_utc,
            "decision_cutoff_hkt": request.decision_cutoff_hkt,
            "selected_run_initialisation_utc": selected_run,
            "selected_run_available_utc": (
                request.selected_run_available_utc
            ),
            "selected_run_key": request.selected_run_key,
            "selected_run_parameter": (
                request.selected_run_parameter
            ),
            "lead_hours_to_event_start": (
                request.lead_hours_to_event_start
            ),
            "lead_hours_to_event_end": (
                request.lead_hours_to_event_end
            ),
            "n_hourly_rows": n_rows,
            "n_nonmissing_temperature_rows": n_nonmissing,
            "n_unique_local_hours": n_unique_hours,
            "forecast_path_ready": ready,
            "forecast_daily_max_c": forecast_max,
            "forecast_daily_min_c": forecast_min,
            "forecast_daily_mean_c": forecast_mean,
            "forecast_max_time_hkt": max_time_hkt,
            "forecast_max_time_utc": max_time_utc,
            "hko_daily_max_c": hko_actual,
            "forecast_error_c": forecast_error,
            "absolute_error_c": (
                abs(forecast_error)
                if ready
                else np.nan
            ),
            "squared_error_c2": (
                forecast_error ** 2
                if ready
                else np.nan
            ),
            "underforecast_indicator": (
                int(forecast_error < 0)
                if ready
                else pd.NA
            ),
        }
    )

    if not ready:
        issue_rows.append(
            {
                "issue_level": "date_rule",
                "issue_code": "DAILY_PATH_NOT_READY",
                "event_date": event_date.date().isoformat(),
                "decision_rule": request.decision_rule,
                "selected_run_initialisation_utc": (
                    selected_run.isoformat()
                ),
                "detail": (
                    f"hourly_rows={n_rows}; "
                    f"nonmissing={n_nonmissing}; "
                    f"unique_local_hours={n_unique_hours}"
                ),
                "blocking": False,
            }
        )

request_hourly = pd.concat(
    request_hourly_frames,
    ignore_index=True,
) if request_hourly_frames else pd.DataFrame(
    columns=[
        "event_date",
        "decision_rule",
        "decision_rule_order",
        "decision_cutoff_utc",
        "decision_cutoff_hkt",
        *run_hourly.columns,
    ]
)

daily_max = pd.DataFrame(daily_rows).sort_values(
    ["event_date", "decision_rule_order"]
).reset_index(drop=True)

if len(daily_max) != 120:
    raise AssertionError(
        f"Expected 120 date-rule daily rows, found {len(daily_max)}"
    )
if daily_max.duplicated(
    ["event_date", "decision_rule"]
).any():
    raise AssertionError(
        "Duplicate date-rule daily forecast keys"
    )

print(
    "Ready date-rule forecasts: "
    f"{int(daily_max['forecast_path_ready'].sum())}/120"
)
display(
    daily_max.groupby("decision_rule")
    .agg(
        requests=("event_date", "size"),
        ready=("forecast_path_ready", "sum"),
        mean_error_c=("forecast_error_c", "mean"),
        mae_c=("absolute_error_c", "mean"),
    )
    .reindex(DECISION_RULE_ORDER)
    .reset_index()
)

Ready date-rule forecasts: 119/120


,decision_rule,requests,ready,mean_error_c,mae_c
0,24h_prior,30,30,-1.813333,1.873333
1,12h_prior,30,30,-1.750000,1.750000
2,6h_prior,30,29,-1.655172,1.758621
3,event_day_open,30,30,-1.656667,1.750000


In [6]:
contract_membership = contract_outcomes.merge(
    daily_max[
        [
            "event_date",
            "decision_rule",
            "decision_rule_order",
            "decision_cutoff_utc",
            "selected_run_initialisation_utc",
            "selected_run_key",
            "forecast_path_ready",
            "forecast_daily_max_c",
            "hko_daily_max_c",
            "forecast_error_c",
            "absolute_error_c",
        ]
    ],
    on=["event_date", "hko_daily_max_c"],
    how="left",
    validate="many_to_many",
)

if len(contract_membership) != 1320:
    raise AssertionError(
        "Expected 1,320 contract-rule membership rows, "
        f"found {len(contract_membership)}"
    )

def membership_for_row(row: pd.Series) -> Any:
    if not bool(row["forecast_path_ready"]):
        return pd.NA
    return deterministic_membership(
        temperature=float(row["forecast_daily_max_c"]),
        event_type=str(row["event_type"]),
        lower_bound=(
            None
            if pd.isna(row["lower_bound_c"])
            else float(row["lower_bound_c"])
        ),
        upper_bound=(
            None
            if pd.isna(row["upper_bound_c"])
            else float(row["upper_bound_c"])
        ),
    )

contract_membership[
    "deterministic_forecast_event_indicator"
] = contract_membership.apply(
    membership_for_row,
    axis=1,
).astype("Int64")

ready_membership = contract_membership.loc[
    contract_membership["forecast_path_ready"]
].copy()

membership_checks = (
    ready_membership.groupby(
        ["event_date", "decision_rule"],
        as_index=False,
    )
    .agg(
        n_contracts=("market_id", "size"),
        n_predicted_yes=(
            "deterministic_forecast_event_indicator",
            "sum",
        ),
        n_realised_yes=("realised_yes", "sum"),
        exact_contract_hit=(
            "deterministic_forecast_event_indicator",
            lambda values: False,
        ),
    )
)

hit_lookup = (
    ready_membership.assign(
        predicted_and_realised=(
            ready_membership[
                "deterministic_forecast_event_indicator"
            ].astype(int)
            * ready_membership["realised_yes"].astype(int)
        )
    )
    .groupby(["event_date", "decision_rule"])[
        "predicted_and_realised"
    ]
    .sum()
    .astype(int)
)

membership_checks["exact_contract_hit"] = [
    bool(hit_lookup.loc[(row.event_date, row.decision_rule)] == 1)
    for row in membership_checks.itertuples(index=False)
]

if not membership_checks["n_contracts"].eq(11).all():
    raise AssertionError(
        "A ready date-rule membership book does not contain 11 contracts"
    )
if not membership_checks["n_predicted_yes"].eq(1).all():
    raise AssertionError(
        "A deterministic forecast does not select exactly one contract"
    )
if not membership_checks["n_realised_yes"].eq(1).all():
    raise AssertionError(
        "A ready date-rule book does not contain exactly one realised winner"
    )

forecast_error_summary_rows: list[dict[str, Any]] = []

for decision_rule in DECISION_RULE_ORDER:
    subset = daily_max.loc[
        daily_max["decision_rule"].eq(decision_rule)
        & daily_max["forecast_path_ready"]
    ].copy()
    rule_hits = membership_checks.loc[
        membership_checks["decision_rule"].eq(decision_rule)
    ]

    if subset.empty:
        forecast_error_summary_rows.append(
            {
                "decision_rule": decision_rule,
                "decision_rule_order": (
                    DECISION_RULE_ORDER.index(decision_rule)
                ),
                "n_ready_dates": 0,
                "mean_forecast_daily_max_c": np.nan,
                "mean_hko_daily_max_c": np.nan,
                "mean_error_c": np.nan,
                "mae_c": np.nan,
                "rmse_c": np.nan,
                "median_absolute_error_c": np.nan,
                "underforecast_rate": np.nan,
                "exact_contract_hit_rate": np.nan,
            }
        )
        continue

    forecast_error_summary_rows.append(
        {
            "decision_rule": decision_rule,
            "decision_rule_order": (
                DECISION_RULE_ORDER.index(decision_rule)
            ),
            "n_ready_dates": len(subset),
            "mean_forecast_daily_max_c": subset[
                "forecast_daily_max_c"
            ].mean(),
            "mean_hko_daily_max_c": subset[
                "hko_daily_max_c"
            ].mean(),
            "mean_error_c": subset[
                "forecast_error_c"
            ].mean(),
            "mae_c": subset["absolute_error_c"].mean(),
            "rmse_c": math.sqrt(
                subset["squared_error_c2"].mean()
            ),
            "median_absolute_error_c": subset[
                "absolute_error_c"
            ].median(),
            "underforecast_rate": pd.to_numeric(
                subset["underforecast_indicator"],
                errors="coerce",
            ).mean(),
            "exact_contract_hit_rate": rule_hits[
                "exact_contract_hit"
            ].mean(),
        }
    )

forecast_error_summary = pd.DataFrame(
    forecast_error_summary_rows
)

display(forecast_error_summary)

,decision_rule,decision_rule_order,n_ready_dates,mean_forecast_daily_max_c,mean_hko_daily_max_c,mean_error_c,mae_c,rmse_c,median_absolute_error_c,underforecast_rate,exact_contract_hit_rate
0,24h_prior,0,30,29.360000,31.173333,-1.813333,1.873333,2.097935,1.75,0.933333,0.033333
1,12h_prior,1,30,29.423333,31.173333,-1.750000,1.750000,1.923105,1.65,1.000000,0.100000
2,6h_prior,2,29,29.462069,31.117241,-1.655172,1.758621,1.978854,1.70,0.931034,0.103448
3,event_day_open,3,30,29.516667,31.173333,-1.656667,1.750000,2.011384,1.55,0.966667,0.066667


In [7]:
integrity_rows: list[dict[str, Any]] = []

def add_check(
    check: str,
    passed: bool,
    detail: str,
    blocking: bool,
) -> None:
    integrity_rows.append(
        {
            "check": check,
            "passed": bool(passed),
            "detail": detail,
            "blocking": bool(blocking),
        }
    )

add_check(
    "request_plan_has_120_date_rule_rows",
    len(request_plan) == 120,
    f"rows={len(request_plan)}",
    True,
)
add_check(
    "request_plan_has_91_unique_runs",
    len(unique_runs) == 91,
    f"unique_runs={len(unique_runs)}",
    True,
)
add_check(
    "selected_runs_are_admissible_at_cutoff",
    request_plan["run_admissible_at_cutoff"].all(),
    (
        "violations="
        f"{int((~request_plan['run_admissible_at_cutoff']).sum())}"
    ),
    True,
)
add_check(
    "selected_runs_use_six_hour_cycles",
    set(
        request_plan[
            "selected_run_initialisation_utc"
        ].dt.hour.unique()
    ).issubset({0, 6, 12, 18}),
    (
        "hours="
        + ",".join(
            map(
                str,
                sorted(
                    request_plan[
                        "selected_run_initialisation_utc"
                    ].dt.hour.unique()
                ),
            )
        )
    ),
    True,
)
add_check(
    "fetch_inventory_has_91_rows",
    len(fetch_inventory) == 91,
    f"rows={len(fetch_inventory)}",
    True,
)
add_check(
    "raw_archive_has_91_files",
    len(list(RAW_DIR.glob("*.json"))) == 91,
    f"files={len(list(RAW_DIR.glob('*.json')))}",
    True,
)
add_check(
    "daily_max_panel_has_120_rows",
    len(daily_max) == 120,
    f"rows={len(daily_max)}",
    True,
)
add_check(
    "daily_max_keys_are_unique",
    not daily_max.duplicated(
        ["event_date", "decision_rule"]
    ).any(),
    (
        "duplicates="
        f"{int(daily_max.duplicated(['event_date', 'decision_rule']).sum())}"
    ),
    True,
)
add_check(
    "successful_fetches_have_parsed_paths",
    (
        fetch_inventory.loc[
            fetch_inventory["success"]
        ]["parse_success"].all()
        if fetch_inventory["success"].any()
        else False
    ),
    (
        "successful_fetches="
        f"{int(fetch_inventory['success'].sum())}; "
        "parsed_successfully="
        f"{int(fetch_inventory['parse_success'].sum())}"
    ),
    False,
)
add_check(
    "at_least_one_ready_date_rule_path",
    daily_max["forecast_path_ready"].any(),
    (
        "ready="
        f"{int(daily_max['forecast_path_ready'].sum())}"
    ),
    False,
)
add_check(
    "ready_paths_have_24_local_hours",
    daily_max.loc[
        daily_max["forecast_path_ready"],
        "n_unique_local_hours",
    ].eq(24).all(),
    (
        "bad_ready_paths="
        f"{int((~daily_max.loc[daily_max['forecast_path_ready'], 'n_unique_local_hours'].eq(24)).sum())}"
    ),
    True,
)
add_check(
    "ready_paths_have_finite_daily_maxima",
    np.isfinite(
        daily_max.loc[
            daily_max["forecast_path_ready"],
            "forecast_daily_max_c",
        ]
    ).all(),
    (
        "bad_ready_maxima="
        f"{int((~np.isfinite(daily_max.loc[daily_max['forecast_path_ready'], 'forecast_daily_max_c'])).sum())}"
    ),
    True,
)
add_check(
    "ready_paths_select_one_contract",
    membership_checks["n_predicted_yes"].eq(1).all(),
    (
        "bad_books="
        f"{int((~membership_checks['n_predicted_yes'].eq(1)).sum())}"
    ),
    True,
)
add_check(
    "no_gaussian_bridge_columns",
    not any(
        "gaussian" in column.lower()
        or column.lower().startswith("p_ecmwf_proxy")
        for column in contract_membership.columns
    ),
    "deterministic membership only",
    True,
)

integrity_checks = pd.DataFrame(integrity_rows)

blocking_failures = integrity_checks.loc[
    integrity_checks["blocking"]
    & ~integrity_checks["passed"]
]
ready_count = int(
    daily_max["forecast_path_ready"].sum()
)

if not blocking_failures.empty:
    verdict = "NEEDS_CORRECTION"
elif ready_count == 120:
    verdict = "PASS"
elif ready_count > 0:
    verdict = "USABLE_WITH_LIMITATIONS"
else:
    verdict = "NEEDS_CORRECTION"

issues = pd.DataFrame(
    issue_rows,
    columns=ISSUE_COLUMNS,
)

print(f"Verdict: {verdict}")
print(f"Ready date-rule paths: {ready_count}/120")
display(integrity_checks)
if not issues.empty:
    display(
        issues.groupby(
            ["issue_level", "issue_code"],
            dropna=False,
        )
        .size()
        .rename("count")
        .reset_index()
    )

Verdict: USABLE_WITH_LIMITATIONS
Ready date-rule paths: 119/120


,check,passed,detail,blocking
0,request_plan_has_120_date_rule_rows,True,rows=120,True
1,request_plan_has_91_unique_runs,True,unique_runs=91,True
2,selected_runs_are_admissible_at_cutoff,True,violations=0,True
3,selected_runs_use_six_hour_cycles,True,"hours=0,6,18",True
4,fetch_inventory_has_91_rows,True,rows=91,True
5,raw_archive_has_91_files,True,files=91,True
6,daily_max_panel_has_120_rows,True,rows=120,True
7,daily_max_keys_are_unique,True,duplicates=0,True
8,successful_fetches_have_parsed_paths,False,successful_fetches=91; parsed_successfully=90,False
9,at_least_one_ready_date_rule_path,True,ready=119,False


,issue_level,issue_code,count
0,date_rule,DAILY_PATH_NOT_READY,1
1,unique_run,RUN_PARSE_FAILED,1


In [8]:
# Convert timezone-aware values to stable ISO strings before writing.
frames_and_time_columns = [
    (
        request_plan,
        [
            "decision_cutoff_utc",
            "decision_cutoff_hkt",
            "selected_run_initialisation_utc",
            "selected_run_available_utc",
            "event_start_hkt",
            "event_end_hkt",
            "event_start_utc",
            "event_end_utc",
        ],
    ),
    (
        fetch_inventory,
        ["selected_run_initialisation_utc"],
    ),
    (
        run_hourly,
        [
            "forecast_valid_time_hkt",
            "forecast_valid_time_utc",
            "selected_run_initialisation_utc",
        ],
    ),
    (
        request_hourly,
        [
            "decision_cutoff_utc",
            "decision_cutoff_hkt",
            "forecast_valid_time_hkt",
            "forecast_valid_time_utc",
            "selected_run_initialisation_utc",
        ],
    ),
    (
        daily_max,
        [
            "decision_cutoff_utc",
            "decision_cutoff_hkt",
            "selected_run_initialisation_utc",
            "selected_run_available_utc",
            "forecast_max_time_hkt",
            "forecast_max_time_utc",
        ],
    ),
    (
        contract_membership,
        [
            "decision_cutoff_utc",
            "selected_run_initialisation_utc",
        ],
    ),
]

for frame, time_columns in frames_and_time_columns:
    if "event_date" in frame.columns:
        frame["event_date"] = pd.to_datetime(
            frame["event_date"]
        ).dt.date.astype(str)
    for column in time_columns:
        if column in frame.columns:
            frame[column] = frame[column].astype(str)

output_paths = {
    "request_plan": (
        OUT_DIR
        / "18q_june_2026_weather_request_plan.csv"
    ),
    "fetch_inventory": (
        OUT_DIR
        / "18q_june_2026_unique_run_fetch_inventory.csv"
    ),
    "run_hourly": (
        OUT_DIR
        / "18q_june_2026_unique_run_hourly_temperature.csv"
    ),
    "request_hourly": (
        OUT_DIR
        / "18q_june_2026_request_hourly_paths.csv"
    ),
    "daily_max": (
        OUT_DIR
        / "18q_june_2026_daily_max_forecasts.csv"
    ),
    "contract_membership": (
        OUT_DIR
        / "18q_june_2026_contract_event_membership.csv"
    ),
    "forecast_error_summary": (
        OUT_DIR
        / "18q_june_2026_forecast_error_summary.csv"
    ),
    "issues": (
        OUT_DIR
        / "18q_june_2026_weather_issues.csv"
    ),
    "integrity_checks": (
        OUT_DIR
        / "18q_june_2026_integrity_checks.csv"
    ),
}

request_plan.to_csv(
    output_paths["request_plan"],
    index=False,
)
fetch_inventory.to_csv(
    output_paths["fetch_inventory"],
    index=False,
)
run_hourly.to_csv(
    output_paths["run_hourly"],
    index=False,
)
request_hourly.to_csv(
    output_paths["request_hourly"],
    index=False,
)
daily_max.to_csv(
    output_paths["daily_max"],
    index=False,
)
contract_membership.to_csv(
    output_paths["contract_membership"],
    index=False,
)
forecast_error_summary.to_csv(
    output_paths["forecast_error_summary"],
    index=False,
)
issues.to_csv(
    output_paths["issues"],
    index=False,
)
integrity_checks.to_csv(
    output_paths["integrity_checks"],
    index=False,
)

ready_daily = daily_max.loc[
    daily_max["forecast_path_ready"]
]

summary = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": verdict,
    "input_dates": 30,
    "input_contracts": 330,
    "decision_rules": DECISION_RULE_ORDER,
    "request_plan_rows": int(len(request_plan)),
    "unique_selected_runs": int(len(unique_runs)),
    "availability_lag_hours": (
        CONSERVATIVE_AVAILABILITY_LAG_HOURS
    ),
    "fetch_inventory_rows": int(len(fetch_inventory)),
    "fetch_http_200_rows": int(
        fetch_inventory["status_code"].eq(200).sum()
    ),
    "fetch_success_rows": int(
        fetch_inventory["success"].sum()
    ),
    "parse_success_rows": int(
        fetch_inventory["parse_success"].sum()
    ),
    "unique_run_hourly_rows": int(len(run_hourly)),
    "request_hourly_rows": int(len(request_hourly)),
    "daily_max_rows": int(len(daily_max)),
    "ready_daily_max_rows": ready_count,
    "unready_daily_max_rows": int(120 - ready_count),
    "ready_dates_any_rule": int(
        ready_daily["event_date"].nunique()
    ),
    "contract_membership_rows": int(
        len(contract_membership)
    ),
    "ready_contract_membership_rows": int(
        len(
            contract_membership.loc[
                contract_membership[
                    "forecast_path_ready"
                ]
            ]
        )
    ),
    "blocking_integrity_failures": int(
        len(blocking_failures)
    ),
    "issue_rows": int(len(issues)),
    "model": MODEL,
    "coordinates": {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
    },
    "hourly_variable": HOURLY_VARIABLE,
    "timezone": TIMEZONE_NAME,
    "forecast_days": FORECAST_DAYS,
    "elevation_override": None,
    "probability_bridge_applied": False,
    "interpretation": (
        "Deterministic ECMWF event-day maxima reconstructed from "
        "issue-time-admissible single runs. Missing paths remain explicit."
    ),
}

summary_path = (
    OUT_DIR
    / "18q_june_2026_weather_ingestion_summary.json"
)
summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

environment = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "requests": requests.__version__,
    "single_runs_api_url": SINGLE_RUNS_URL,
    "model": MODEL,
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "hourly_variable": HOURLY_VARIABLE,
    "timezone": TIMEZONE_NAME,
    "forecast_days": FORECAST_DAYS,
    "run_frequency_hours": RUN_FREQUENCY_HOURS,
    "availability_lag_hours": (
        CONSERVATIVE_AVAILABILITY_LAG_HOURS
    ),
    "cell_selection": "land",
    "elevation_override": None,
}
environment_path = (
    OUT_DIR
    / "18q_june_2026_environment.json"
)
environment_path.write_text(
    json.dumps(
        environment,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "step": "18q",
  "generated_at_utc": "2026-07-21T04:59:00.893049+00:00",
  "verdict": "USABLE_WITH_LIMITATIONS",
  "input_dates": 30,
  "input_contracts": 330,
  "decision_rules": [
    "24h_prior",
    "12h_prior",
    "6h_prior",
    "event_day_open"
  ],
  "request_plan_rows": 120,
  "unique_selected_runs": 91,
  "availability_lag_hours": 6,
  "fetch_inventory_rows": 91,
  "fetch_http_200_rows": 91,
  "fetch_success_rows": 91,
  "parse_success_rows": 90,
  "unique_run_hourly_rows": 21600,
  "request_hourly_rows": 2856,
  "daily_max_rows": 120,
  "ready_daily_max_rows": 119,
  "unready_daily_max_rows": 1,
  "ready_dates_any_rule": 30,
  "contract_membership_rows": 1320,
  "ready_contract_membership_rows": 1309,
  "blocking_integrity_failures": 0,
  "issue_rows": 2,
  "model": "ecmwf_ifs",
  "coordinates": {
    "latitude": 22.302219,
    "longitude": 114.174637
  },
  "hourly_variable": "temperature_2m",
  "timezone": "Asia/Hong_Kong",
  "forecast_days": 10,
  "elevation_override

In [9]:
report_lines = [
    "# 18q June 2026 ECMWF IFS single-run forecast ingestion",
    "",
    f"Generated at UTC: `{summary['generated_at_utc']}`",
    "",
    "## Overall judgement",
    "",
    f"**{summary['verdict']}**",
    "",
    "## Method",
    "",
    (
        "The latest ECMWF IFS run satisfying the conservative "
        "availability condition `run initialisation + 6 hours <= "
        "decision time` was selected for each date-rule request."
    ),
    "",
    (
        "The Open-Meteo Single Runs API was queried at the fixed "
        "Hong Kong Observatory coordinates using hourly two-metre "
        "temperature in Asia/Hong_Kong time. The event-day maximum "
        "is the maximum of the 24 local hourly values."
    ),
    "",
    (
        "No elevation override, observation substitution, later-run "
        "replacement or Gaussian probability bridge was used."
    ),
    "",
    "## Sample flow",
    "",
    f"- Input June dates: {summary['input_dates']}",
    f"- Input certified contracts: {summary['input_contracts']}",
    f"- Date-rule requests: {summary['request_plan_rows']}",
    f"- Unique selected runs: {summary['unique_selected_runs']}",
    f"- Successful run fetches: {summary['fetch_success_rows']}",
    f"- Successfully parsed runs: {summary['parse_success_rows']}",
    f"- Unique-run hourly rows: {summary['unique_run_hourly_rows']}",
    f"- Request-level hourly rows: {summary['request_hourly_rows']}",
    f"- Ready daily maximum forecasts: {summary['ready_daily_max_rows']}",
    f"- Unready daily maximum forecasts: {summary['unready_daily_max_rows']}",
    f"- Issue rows: {summary['issue_rows']}",
    "",
    "## Forecast error by decision rule",
    "",
    (
        "| Rule | Ready dates | Mean forecast maximum | "
        "Mean HKO maximum | Mean error | MAE | RMSE | "
        "Underforecast rate | Exact contract hit rate |"
    ),
    "|---|---:|---:|---:|---:|---:|---:|---:|---:|",
]

for row in forecast_error_summary.sort_values(
    "decision_rule_order"
).itertuples(index=False):
    def fmt(value: Any, digits: int = 6) -> str:
        if pd.isna(value):
            return ""
        return f"{float(value):.{digits}f}"

    report_lines.append(
        "| {rule} | {n} | {forecast} | {hko} | {error} | "
        "{mae} | {rmse} | {under} | {hit} |".format(
            rule=row.decision_rule,
            n=int(row.n_ready_dates),
            forecast=fmt(row.mean_forecast_daily_max_c),
            hko=fmt(row.mean_hko_daily_max_c),
            error=fmt(row.mean_error_c),
            mae=fmt(row.mae_c),
            rmse=fmt(row.rmse_c),
            under=fmt(row.underforecast_rate),
            hit=fmt(row.exact_contract_hit_rate),
        )
    )

report_lines.extend(
    [
        "",
        "## Availability by decision rule",
        "",
        "| Rule | Requests | Ready | Missing |",
        "|---|---:|---:|---:|",
    ]
)

availability = (
    daily_max.groupby("decision_rule")
    .agg(
        requests=("event_date", "size"),
        ready=("forecast_path_ready", "sum"),
    )
    .reindex(DECISION_RULE_ORDER)
)
availability["missing"] = (
    availability["requests"]
    - availability["ready"]
)

for rule, row in availability.iterrows():
    report_lines.append(
        f"| {rule} | {int(row['requests'])} | "
        f"{int(row['ready'])} | {int(row['missing'])} |"
    )

report_lines.extend(
    [
        "",
        "## Integrity checks",
        "",
        "| Check | Passed | Blocking | Detail |",
        "|---|---|---|---|",
    ]
)

for row in integrity_checks.itertuples(index=False):
    report_lines.append(
        f"| {row.check} | {row.passed} | "
        f"{row.blocking} | {row.detail} |"
    )

report_lines.extend(
    [
        "",
        "## Downstream use",
        "",
        (
            "Only ready date-rule rows may enter the expanded "
            "market-versus-weather common-support panel. The deterministic "
            "forecast maximum is an input to subsequent local residual "
            "post-processing; it is not itself treated as a calibrated "
            "probability distribution."
        ),
    ]
)

report_path = (
    REPORT_DIR
    / "18q_june_2026_ecmwf_single_run_forecast_report.md"
)
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

# Build the manifest after all canonical files are final.
manifest_rows: list[dict[str, Any]] = []
for root in (RAW_DIR, OUT_DIR, REPORT_DIR):
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "18q_june_2026_sha256_manifest.csv":
            continue
        manifest_rows.append(
            {
                "path": str(path.relative_to(REPO_ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

manifest_path = (
    OUT_DIR
    / "18q_june_2026_sha256_manifest.csv"
)
pd.DataFrame(manifest_rows).to_csv(
    manifest_path,
    index=False,
)

print(f"Report: {report_path.relative_to(REPO_ROOT)}")
print(f"Manifest entries: {len(manifest_rows)}")

Report: reports/18q_june_2026_ecmwf_single_run_forecasts/18q_june_2026_ecmwf_single_run_forecast_report.md
Manifest entries: 103


In [10]:
print("Availability by rule:")
display(
    daily_max.groupby("decision_rule")
    .agg(
        requests=("event_date", "size"),
        ready=("forecast_path_ready", "sum"),
        mean_error_c=("forecast_error_c", "mean"),
        mae_c=("absolute_error_c", "mean"),
    )
    .reindex(DECISION_RULE_ORDER)
    .assign(
        missing=lambda frame: (
            frame["requests"] - frame["ready"]
        )
    )
    .reset_index()
)

print("Forecast error summary:")
display(forecast_error_summary)

print("Issue summary:")
if issues.empty:
    print("No issue rows.")
else:
    display(
        issues.groupby(
            ["issue_level", "issue_code"],
            dropna=False,
        )
        .size()
        .rename("count")
        .reset_index()
    )

print(f"Final verdict: {verdict}")

Availability by rule:


,decision_rule,requests,ready,mean_error_c,mae_c,missing
0,24h_prior,30,30,-1.813333,1.873333,0
1,12h_prior,30,30,-1.750000,1.750000,0
2,6h_prior,30,29,-1.655172,1.758621,1
3,event_day_open,30,30,-1.656667,1.750000,0


Forecast error summary:


,decision_rule,decision_rule_order,n_ready_dates,mean_forecast_daily_max_c,mean_hko_daily_max_c,mean_error_c,mae_c,rmse_c,median_absolute_error_c,underforecast_rate,exact_contract_hit_rate
0,24h_prior,0,30,29.360000,31.173333,-1.813333,1.873333,2.097935,1.75,0.933333,0.033333
1,12h_prior,1,30,29.423333,31.173333,-1.750000,1.750000,1.923105,1.65,1.000000,0.100000
2,6h_prior,2,29,29.462069,31.117241,-1.655172,1.758621,1.978854,1.70,0.931034,0.103448
3,event_day_open,3,30,29.516667,31.173333,-1.656667,1.750000,2.011384,1.55,0.966667,0.066667


Issue summary:


,issue_level,issue_code,count
0,date_rule,DAILY_PATH_NOT_READY,1
1,unique_run,RUN_PARSE_FAILED,1


Final verdict: USABLE_WITH_LIMITATIONS
